# Construction Site Operations — Pre-Filled Configuration
### Ready-to-Run Config for the Construction Use Case

This is a **pre-configured** version of `00_Industry_Config` specifically for the
**Construction Site Documentation Burden** demo. All table lists, artifact names,
and data paths are hardcoded from the existing 23-CSV construction dataset.

**Use this instead of `00_Industry_Config` if you want to skip auto-discovery and run
directly with the known construction tables.**

---

### Data Summary
- **6 Dimensions** — Supervisors, Projects, Project Sites, Inspection Types, Subcontractors, Trade Phases
- **6 Batch Facts** — Daily Logs, Safety Inspections, Supervisor Wellness, Inspection Quality, Subcontractor Satisfaction, Project Performance
- **6 Event Facts** → Eventhouse — PM Interactions, RFI Events, Change Orders, Phase Handoffs, Safety Alerts, Site Location
- **5 Streams** → Eventhouse — Daily Log Activity, Safety Events, RFI Status, Weather Conditions, Equipment Telemetry

### Ontology
- **6 Entity Types:** Supervisor, Project, ProjectSite, InspectionType, Subcontractor, TradePhase
- **5 Relationship Types:** supervises_project, located_at, assigned_inspection, contracted_for, follows_phase
- **6 Contextualizations:** DailyLog, SafetyInspection, PMInteraction, RFIEvent, ChangeOrder, SafetyAlert

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INDUSTRY SETTING
# ════════════════════════════════════════════════════════════════════════

INDUSTRY       = "construction"
INDUSTRY_LABEL = "Construction"

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# FABRIC ARTIFACT NAMES
# ════════════════════════════════════════════════════════════════════════
# Update these to match your Fabric workspace artifact names.

LAKEHOUSE_NAME      = "ConstructionLakehouse"
WAREHOUSE_NAME      = "Construction_Datawarehouse"
EVENTHOUSE_NAME     = "construction_rt_store"
ONTOLOGY_NAME       = "ConstructionSiteOpsOntology"
DATA_AGENT_NAME     = "ConstructionQA"
OPS_AGENT_NAME      = "ConstructionDocBurden"
SEMANTIC_MODEL_NAME = "Construction_Site_Ops_Model"

print(f"Lakehouse:      {LAKEHOUSE_NAME}")
print(f"Warehouse:      {WAREHOUSE_NAME}")
print(f"Eventhouse:     {EVENTHOUSE_NAME}")
print(f"Ontology:       {ONTOLOGY_NAME}")
print(f"Data Agent:     {DATA_AGENT_NAME}")
print(f"Ops Agent:      {OPS_AGENT_NAME}")
print(f"Semantic Model: {SEMANTIC_MODEL_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DATA PATHS & EVENTHOUSE CONNECTION
# ════════════════════════════════════════════════════════════════════════

# CSV files location in Lakehouse Files area
CSV_BASE_PATH = "/lakehouse/default/Files/construction_site_operations/data"

# Schemas
LAKEHOUSE_SCHEMA = "dbo"
WAREHOUSE_SCHEMA = "dbo"

# ── Eventhouse Connection ───────────────────────────────────────────────
# Fill these in from your Fabric workspace
EVENTHOUSE_CLUSTER_URI = ""   # e.g. "https://trd-xxxxx.z5.kusto.fabric.microsoft.com"
EVENTHOUSE_DATABASE    = ""   # e.g. "construction_rt_store"

# Ontology package path
ONTOLOGY_PACKAGE_PATH = "/lakehouse/default/Files/construction_site_ops_ontology_package.iq"

print(f"CSV source:     {CSV_BASE_PATH}")
print(f"LH schema:      {LAKEHOUSE_SCHEMA}")
print(f"WH schema:      {WAREHOUSE_SCHEMA}")
print(f"Eventhouse:     {EVENTHOUSE_CLUSTER_URI or '(fill in your cluster URI)'}")
print(f"Ontology pkg:   {ONTOLOGY_PACKAGE_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CONSTRUCTION TABLE DEFINITIONS (pre-filled, no auto-discovery needed)
# ════════════════════════════════════════════════════════════════════════

import os, json

# ── 6 Dimension Tables → Lakehouse + Warehouse ──────────────────────────
DIM_TABLES = [
    "dim_supervisors",
    "dim_projects",
    "dim_project_sites",
    "dim_inspection_types",
    "dim_subcontractors",
    "dim_trade_phases",
]

# ── 6 Batch Fact Tables → Lakehouse + Warehouse ─────────────────────────
FACT_TABLES_BATCH = [
    "fact_daily_logs",
    "fact_safety_inspections",
    "fact_supervisor_wellness",
    "fact_inspection_quality",
    "fact_subcontractor_satisfaction",
    "fact_project_performance",
]

# ── 6 Event Fact Tables → Eventhouse (KQL) ──────────────────────────────
FACT_TABLES_EVENT = [
    "fact_pm_interactions",
    "fact_rfi_events",
    "fact_change_orders",
    "fact_phase_handoffs",
    "fact_safety_alerts",
    "fact_site_location",
]

# ── 5 Streaming Tables → Eventhouse (KQL) ───────────────────────────────
STREAM_TABLES = [
    "stream_daily_log_activity",
    "stream_safety_events",
    "stream_rfi_status",
    "stream_weather_conditions",
    "stream_equipment_telemetry",
]

# ── Combined Target Lists ───────────────────────────────────────────────
LAKEHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH
WAREHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT
EVENTHOUSE_TABLES = FACT_TABLES_EVENT + STREAM_TABLES

# ── KQL Table Name Mapping (CSV name → KQL table name) ─────────────────
# Event facts strip the 'fact_' prefix; streams strip 'stream_' prefix
EVENTHOUSE_KQL_NAMES = {
    "fact_pm_interactions":          "pm_interactions",
    "fact_rfi_events":               "rfi_events",
    "fact_change_orders":            "change_orders",
    "fact_phase_handoffs":           "phase_handoffs",
    "fact_safety_alerts":            "safety_alerts",
    "fact_site_location":            "site_location",
    "stream_daily_log_activity":     "daily_log_activity",
    "stream_safety_events":          "safety_events",
    "stream_rfi_status":             "rfi_status",
    "stream_weather_conditions":     "weather_conditions",
    "stream_equipment_telemetry":    "equipment_telemetry",
}

print(f"{'='*60}")
print(f"CONSTRUCTION TABLE INVENTORY")
print(f"{'='*60}")
print(f"\nDimension tables ({len(DIM_TABLES)}):")
for t in DIM_TABLES: print(f"  • {t}")
print(f"\nFact tables — Batch ({len(FACT_TABLES_BATCH)}):")
for t in FACT_TABLES_BATCH: print(f"  • {t}")
print(f"\nFact tables — Event/Eventhouse ({len(FACT_TABLES_EVENT)}):")
for t in FACT_TABLES_EVENT: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\nStreaming tables — Eventhouse ({len(STREAM_TABLES)}):")
for t in STREAM_TABLES: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\n{'─'*60}")
print(f"Lakehouse target:  {len(LAKEHOUSE_TABLES)} tables (12 Delta tables)")
print(f"Warehouse target:  {len(WAREHOUSE_TABLES)} tables (18 SQL tables)")
print(f"Eventhouse target: {len(EVENTHOUSE_TABLES)} tables (11 KQL tables)")
print(f"Total:             23 CSV files")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# EXPECTED ROW COUNTS (for validation in downstream notebooks)
# ════════════════════════════════════════════════════════════════════════

EXPECTED_ROW_COUNTS = {
    # Dimensions
    "dim_supervisors":                25,
    "dim_projects":                   12,
    "dim_project_sites":              12,
    "dim_inspection_types":           20,
    "dim_subcontractors":             30,
    "dim_trade_phases":               25,
    # Batch Facts
    "fact_daily_logs":               200,
    "fact_safety_inspections":       150,
    "fact_supervisor_wellness":       25,
    "fact_inspection_quality":       150,
    "fact_subcontractor_satisfaction": 30,
    "fact_project_performance":       48,
    # Event Facts
    "fact_pm_interactions":          400,
    "fact_rfi_events":                80,
    "fact_change_orders":             40,
    "fact_phase_handoffs":            25,
    "fact_safety_alerts":             60,
    "fact_site_location":            500,
    # Streams
    "stream_daily_log_activity":     100,
    "stream_safety_events":           80,
    "stream_rfi_status":              80,
    "stream_weather_conditions":      80,
    "stream_equipment_telemetry":     80,
}

total_lakehouse = sum(EXPECTED_ROW_COUNTS[t] for t in LAKEHOUSE_TABLES)
total_eventhouse = sum(EXPECTED_ROW_COUNTS[t] for t in EVENTHOUSE_TABLES)
print(f"Expected Lakehouse rows:  {total_lakehouse:,}")
print(f"Expected Eventhouse rows: {total_eventhouse:,}")
print(f"Expected total rows:      {sum(EXPECTED_ROW_COUNTS.values()):,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SCHEMA INSPECTION HELPER
# ════════════════════════════════════════════════════════════════════════

def preview_table(table_name, base_path=CSV_BASE_PATH, rows=5):
    """Read a CSV and display schema + sample rows."""
    path = f"{base_path}/{table_name}.csv"
    df = spark.read.option("header", True).option("inferSchema", True).csv(path)
    print(f"\n{'─'*60}")
    print(f"Table: {table_name}  |  {df.count()} rows  |  {len(df.columns)} columns")
    print(f"{'─'*60}")
    df.printSchema()
    df.show(rows, truncate=False)
    return df

# Uncomment to preview specific tables:
# preview_table("dim_supervisors")
# preview_table("fact_daily_logs")
# preview_table("stream_safety_events")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CONFIG SUMMARY (exported to downstream notebooks via %run)
# ════════════════════════════════════════════════════════════════════════

CONFIG = {
    "industry":             INDUSTRY,
    "industry_label":       INDUSTRY_LABEL,
    "lakehouse_name":       LAKEHOUSE_NAME,
    "warehouse_name":       WAREHOUSE_NAME,
    "eventhouse_name":      EVENTHOUSE_NAME,
    "eventhouse_uri":       EVENTHOUSE_CLUSTER_URI,
    "eventhouse_db":        EVENTHOUSE_DATABASE,
    "ontology_name":        ONTOLOGY_NAME,
    "ontology_package":     ONTOLOGY_PACKAGE_PATH,
    "data_agent_name":      DATA_AGENT_NAME,
    "ops_agent_name":       OPS_AGENT_NAME,
    "semantic_model_name":  SEMANTIC_MODEL_NAME,
    "csv_base_path":        CSV_BASE_PATH,
    "lakehouse_schema":     LAKEHOUSE_SCHEMA,
    "warehouse_schema":     WAREHOUSE_SCHEMA,
    "dim_tables":           DIM_TABLES,
    "fact_tables_batch":    FACT_TABLES_BATCH,
    "fact_tables_event":    FACT_TABLES_EVENT,
    "stream_tables":        STREAM_TABLES,
    "lakehouse_tables":     LAKEHOUSE_TABLES,
    "warehouse_tables":     WAREHOUSE_TABLES,
    "eventhouse_tables":    EVENTHOUSE_TABLES,
}

print(f"\n{'═'*60}")
print(f"✅ CONSTRUCTION CONFIG READY")
print(f"{'═'*60}")
print(json.dumps({k: v if not isinstance(v, list) else f"{len(v)} tables" for k, v in CONFIG.items()}, indent=2))
print(f"\nThis config is imported by downstream notebooks via:")
print(f"  %run ./Construction_Config")